#1: Setup

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

RAW_DIR = Path("..") / "data" / "raw"

#2: load all 9 tables

In [2]:
FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

tables = {name: pd.read_csv(RAW_DIR / f) for name, f in FILES.items()}

#3: overview of every table

In [3]:
overview = pd.DataFrame({
    "rows": {n: df.shape[0] for n, df in tables.items()},
    "columns": {n: df.shape[1] for n, df in tables.items()},
    "total_nulls": {n: int(df.isnull().sum().sum()) for n, df in tables.items()},
    "duplicate_rows": {n: int(df.duplicated().sum()) for n, df in tables.items()},
})
overview

,rows,columns,total_nulls,duplicate_rows
customers,99441,5,0,0
orders,99441,8,4908,0
order_items,112650,7,0,0
payments,103886,5,0,0
reviews,99224,7,145903,0
products,32951,9,2448,0
sellers,3095,4,0,0
geolocation,1000163,5,0,261831
category_translation,71,2,0,0


#4: look inside one table (change the name to explore others)

In [4]:
df = tables["orders"]
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


#5: column types and missing values for that table

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


#6: missing values only

In [6]:
nulls = df.isnull().sum()
nulls[nulls > 0]

order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

#7: The key customer check

In [7]:
c = tables["customers"]
print("Unique customer_id:       ", c["customer_id"].nunique())
print("Unique customer_unique_id:", c["customer_unique_id"].nunique())

Unique customer_id:        99441
Unique customer_unique_id: 96096


#8: Order status breakdown

In [8]:
tables["orders"]["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

#9: where exactly are the order nulls?

In [9]:
orders = tables["orders"]
orders.isnull().sum()[lambda s: s > 0]

order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

#10: do the missing dates match the non-delivered statuses?

In [10]:
date_cols = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]
orders.groupby("order_status")[date_cols].apply(lambda g: g.isnull().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


#11: date columns are probably text. Check, then see the time range

In [11]:
print(orders.dtypes)

purchase = pd.to_datetime(orders["order_purchase_timestamp"])
print(purchase.min(), "->", purchase.max())

purchase.dt.to_period("M").value_counts().sort_index()

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object
2016-09-04 21:15:19 -> 2018-10-17 17:30:18


order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

#12: reviews, is it one review per order

In [12]:
r = tables["reviews"]
print("Rows:", len(r))
print("Unique review_id:", r["review_id"].nunique())
print("Unique order_id: ", r["order_id"].nunique())

Rows: 99224
Unique review_id: 98410
Unique order_id:  98673


#13: payments per order

In [13]:
p = tables["payments"]
print("Rows:", len(p), "| Unique orders:", p["order_id"].nunique())
p["payment_type"].value_counts()

Rows: 103886 | Unique orders: 99440


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

#14: items per order

In [14]:
oi = tables["order_items"]
print("Rows:", len(oi), "| Unique orders:", oi["order_id"].nunique())

Rows: 112650 | Unique orders: 98666


#15: orphan check (keys that don’t match across tables)

In [15]:
o_ids = set(orders["order_id"])
print("Items without an order:   ", (~oi["order_id"].isin(o_ids)).sum())
print("Payments without an order:", (~p["order_id"].isin(o_ids)).sum())
print("Reviews without an order: ", (~r["order_id"].isin(o_ids)).sum())
print("Orders without items:     ", (~orders["order_id"].isin(set(oi["order_id"]))).sum())

Items without an order:    0
Payments without an order: 0
Reviews without an order:  0
Orders without items:      775


#16: geolocation

In [16]:
g = tables["geolocation"]
print("Rows:", len(g), "| Duplicate rows:", g.duplicated().sum())
print("Unique zip prefixes:", g["geolocation_zip_code_prefix"].nunique())

Rows: 1000163 | Duplicate rows: 261831
Unique zip prefixes: 19015


In [17]:
no_items = orders[~orders["order_id"].isin(set(oi["order_id"]))]
no_items["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

In [18]:
orders = tables["orders"]
oi = tables["order_items"]
p = tables["payments"]
r = tables["reviews"]
g = tables["geolocation"]
date_cols = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]

print("=== 1. Columns with nulls ===")
for name in ["orders", "reviews", "products"]:
    n = tables[name].isnull().sum()
    print(f"[{name}]")
    print(n[n > 0].to_string(), "\n")

print("=== 2. Date nulls by order_status ===")
print(orders.groupby("order_status")[date_cols].apply(lambda d: d.isnull().sum()), "\n")

print("=== 3. Orders without items, by status ===")
no_items = orders[~orders["order_id"].isin(set(oi["order_id"]))]
print(no_items["order_status"].value_counts(), "\n")

print("=== 4. One-to-many check (orders with more than 1 row) ===")
print("Orders with >1 item row:   ", (oi.groupby("order_id").size() > 1).sum())
print("Orders with >1 payment row:", (p.groupby("order_id").size() > 1).sum())
print("Orders with >1 review:     ", (r.groupby("order_id").size() > 1).sum(), "\n")

print("=== 5. Reviews keys ===")
print("Rows:", len(r), "| Unique review_id:", r["review_id"].nunique(), "| Unique order_id:", r["order_id"].nunique(), "\n")

print("=== 6. Payment types ===")
print(p["payment_type"].value_counts(), "\n")

print("=== 7. Geolocation ===")
print("Rows:", len(g), "| Duplicate rows:", g.duplicated().sum())
print("Unique zip prefixes:", g["geolocation_zip_code_prefix"].nunique(), "\n")

print("=== 8. Table overview (all 9) ===")
print(overview)

=== 1. Columns with nulls ===
[orders]
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965 

[reviews]
review_comment_title      87656
review_comment_message    58247 

[products]
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2 

=== 2. Date nulls by order_status ===
              order_approved_at  order_delivered_carrier_date  \
order_status                                                    
approved                      0                             2   
canceled                    141                           550   
created                       5                             5   
delivered                    14                             2   
invoiced                      0                         

In [19]:
print(orders[["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]].dtypes, "\n")

purchase = pd.to_datetime(orders["order_purchase_timestamp"])
print("Date range:", purchase.min(), "->", purchase.max(), "\n")

purchase.dt.to_period("M").value_counts().sort_index()

order_purchase_timestamp         str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object 

Date range: 2016-09-04 21:15:19 -> 2018-10-17 17:30:18 



order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

the revenue inflation trap

In [20]:
# The correct way: sum items per order
true_revenue = oi["price"].sum()

# The WRONG way: join items directly to payments
wrong = oi.merge(p, on="order_id")
wrong_revenue = wrong["price"].sum()

print(f"Correct item revenue:   {true_revenue:,.2f}")
print(f"Revenue after naive join: {wrong_revenue:,.2f}")
print(f"Inflation: {wrong_revenue / true_revenue - 1:.1%}")
print(f"Rows: {len(oi):,} before join -> {len(wrong):,} after join")

Correct item revenue:   13,591,643.70
Revenue after naive join: 14,209,115.34
Inflation: 4.5%
Rows: 112,650 before join -> 117,601 after join


do payments reconcile with items?

In [21]:
items_total = (oi.assign(item_total=oi["price"] + oi["freight_value"])
                 .groupby("order_id")["item_total"].sum())
pay_total = p.groupby("order_id")["payment_value"].sum()

compare = pd.concat([items_total, pay_total], axis=1, join="inner")
diff = (compare["item_total"] - compare["payment_value"]).abs()

print("Orders compared:", len(compare))
print("Orders where items+freight differ from payments by more than R$1:", (diff > 1).sum())

Orders compared: 98665
Orders where items+freight differ from payments by more than R$1: 249


three small anomaly checks

In [ ]:
print("--- not_defined payments ---")
print(p[p["payment_type"] == "not_defined"], "\n")

print("--- delivered orders with no delivery date ---")
print(orders[(orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].isnull()][["order_id", "order_status", "order_delivered_carrier_date"]], "\n")

print("--- categories in products missing from the translation table ---")
prod_cats = set(tables["products"]["product_category_name"].dropna())
trans_cats = set(tables["category_translation"]["product_category_name"])
print(prod_cats - trans_cats)